<a href="https://colab.research.google.com/github/SAIKUNDAN1/Capable-Project/blob/main/travel_concierge_ai_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U -q langchain langchain-google-genai pypdf langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.5/346.5 kB 20.7 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.2 MB/s eta 0:00:00


In [ ]:
!pip install -q streamlit
import streamlit as st
st.title("Travel Concierge AI")
user_input = st.text_input("Ask me about the travel documents!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 78.4 MB/s eta 0:00:00


2026-03-05 17:16:13.141 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:16:13.323 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-05 17:16:13.324 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:16:13.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:16:13.326 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:16:13.327 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:16:13.329 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:16:13.330 Thread 'MainThread': mi

In [ ]:
import streamlit as st
import os
import sqlite3
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Set your API Key
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"

In [ ]:
def init_db():
    conn = sqlite3.connect('history.db')
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS searches
                 (id INTEGER PRIMARY KEY, query TEXT, response TEXT)''')
    conn.commit()
    conn.close()

def save_to_db(query, response):
    conn = sqlite3.connect('history.db')
    c = conn.cursor()
    c.execute("INSERT INTO searches (query, response) VALUES (?, ?)", (query, response))
    conn.commit()
    conn.close()

In [ ]:
def process_pdf(pdf_file):
    # Save uploaded file temporarily to disk
    with open("temp.pdf", "wb") as f:
        f.write(pdf_file.getbuffer())

    loader = PyPDFLoader("temp.pdf")
    docs = loader.load()

    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(docs)

    # Create Vector Store
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

In [ ]:
st.title("🚀 Travel Concierge AI")
init_db()

uploaded_file = st.file_uploader("Upload your Travel Docs (PDF)", type="pdf")

if uploaded_file:
    st.success("File Uploaded!")
    vector_store = process_pdf(uploaded_file)

    user_query = st.chat_input("Ask about your trip...")

    if user_query:
        # Initialize LLM
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

        # Create Chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm, chain_type="stuff", retriever=vector_store.as_retriever()
        )

        # Get Answer
        response = qa_chain.invoke(user_query)
        answer = response["result"]

        # Show and Save
        st.write(f"**AI:** {answer}")
        save_to_db(user_query, answer)

2026-03-05 17:30:42.013 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.014 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.015 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.031 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.032 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.033 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.035 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-05 17:30:42.037 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
def display_history():
    with st.sidebar:
        st.header("🕒 Recent Searches")
        conn = sqlite3.connect('history.db')
        c = conn.cursor()
        # Fetch last 5 searches
        c.execute("SELECT query FROM searches ORDER BY id DESC LIMIT 5")
        history = c.fetchall()
        conn.close()

        if history:
            for item in history:
                st.info(f"🔍 {item[0]}")
        else:
            st.write("No history yet!")

In [ ]:
# To install these specific versions, use the following command:
# !pip install streamlit==1.42.0 langchain==0.3.0 langchain-community==0.3.0 langchain-google-genai==2.0.0 faiss-cpu==1.8.0 pypdf==4.0.0 python-dotenv==1.0.1

In [ ]:
%%writefile app.py
import streamlit as st
import os
import sqlite3
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- DATABASE SETUP ---
def init_db():
    conn = sqlite3.connect('travel_history.db')
    c = conn.cursor()
    c.execute('CREATE TABLE IF NOT EXISTS searches (id INTEGER PRIMARY KEY, query TEXT, response TEXT)')
    conn.commit()
    conn.close()

def save_search(query, response):
    conn = sqlite3.connect('travel_history.db')
    c = conn.cursor()
    c.execute("INSERT INTO searches (query, response) VALUES (?, ?)", (query, response))
    conn.commit()
    conn.close()

# --- APP LAYOUT ---
st.set_page_config(page_title="Travel Concierge AI", layout="wide")
init_db()

# Sidebar for History
with st.sidebar:
    st.title("🕒 Search History")
    conn = sqlite3.connect('travel_history.db')
    c = conn.cursor()
    c.execute("SELECT query FROM searches ORDER BY id DESC LIMIT 5")
    for row in c.fetchall():
        st.info(f"🔍 {row[0]}")
    conn.close()

st.title("✈️ AI Travel Concierge")
st.write("Upload your travel documents and ask me anything!")

# API Key handling: Use Colab Secrets if available
try:
    api_key = userdata.get('GOOGLE_API_KEY')
except:
    api_key = os.getenv("GOOGLE_API_KEY")

# --- RAG PIPELINE ---
uploaded_file = st.file_uploader("Upload PDF Documents", type="pdf")

if uploaded_file and api_key:
    with open("temp.pdf", "wb") as f:
        f.write(uploaded_file.getbuffer())

    loader = PyPDFLoader("temp.pdf")
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_documents(docs)

    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)
    vector_store = FAISS.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever()

    user_query = st.chat_input("Ask about your trip...")

    if user_query:
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", google_api_key=api_key)
        template = """You are an AI Travel Concierge. Use context: {context}\nQuestion: {question}"""
        prompt = ChatPromptTemplate.from_template(template)
        def format_docs(docs): return "\n\n".join(d.page_content for d in docs)

        rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt | llm | StrOutputParser()
        )

        answer = rag_chain.invoke(user_query)
        st.chat_message("assistant").write(answer)
        save_search(user_query, answer)
elif not api_key:
    st.warning("Please set GOOGLE_API_KEY in Colab Secrets.")

# TO VIEW THE APP:
# 1. Run this cell to create app.py
# 2. In a new cell run:
# !npm install -g localtunnel
# !streamlit run app.py & npx localtunnel --port 8501

Writing app.py


In [ ]:
GOOGLE_API_KEY = "your-actual-key-here"